**Séance :** 02  
**Nom :** FANOU Précieux  
**Porteur # :** P2  
**Étape assignée :** 2.2 --> 2.5

# Étape 2.5 : Le premier modèle

**Question** : un modèle simple, entraîné sur les 130 variables, bat-il le plancher de l'étape 1.10 de **plus que le bruit** mesuré à l'étape 2.2 ?

| Choix | Ce qu'on retient pour ce premier essai | Pourquoi |
|---|---|---|
| Modèle | `HistGradientBoostingRegressor` (scikit-learn) | arbres boostés, rapides sur CPU, acceptent des milliers de lignes et 130 variables |
| Perte d'apprentissage | `absolute_error` | la métrique est la MAE : on apprend ce qu'on mesure |
| Encodage des `cat` | **codes ordinaux** communs aux deux fichiers | tenable pour 116 colonnes, y compris celles à plus de 255 modalités |
| Validation | le même découpage que l'étape 1.10 (graine 42, stratifié) | pour comparer au même plancher |
| Règle de décision | gain > 2 x barre de bruit (étape 2.2) | un gain dans le bruit n'est pas démontré |

> **Ce choix d'encodage est un premier essai, pas une décision définitive.** Les codes ordinaux imposent un ordre alphabétique sans signification métier. Les modèles suivants (au moins huit essais sont demandés) pourront tester d'autres encodages : fréquence, catégorielles natives pour les colonnes à 255 modalités ou moins, etc.

> **Aucune fuite** : `id` et `loss` ne font pas partie des variables du modèle. Le modèle ne voit jamais la partie cachée pendant son entraînement.

## Bloc 1 : imports, chemins et réglages

In [1]:
# ── Ce que fait ce bloc : importer les outils, trouver les fichiers, fixer les réglages ──
import time                                            # pour mesurer le temps d'entraînement
from pathlib import Path                               # chemins de fichiers
import numpy as np                                     # calcul sur les nombres
import pandas as pd                                    # tableaux
from sklearn.ensemble import HistGradientBoostingRegressor   # le modèle
from sklearn.model_selection import train_test_split         # le découpage de validation

# Sur Kaggle, les données sont dans /kaggle/input/... ; en local, dans le dossier courant
RACINE_KAGGLE = Path("/kaggle/input")

if RACINE_KAGGLE.exists():
    trouves = [p for p in RACINE_KAGGLE.rglob("train.csv") if "sujet-04" in str(p)]
    if not trouves:
        raise FileNotFoundError("train.csv du sujet 04 introuvable : ajoutez votre jeu (Add Input).")
    DOSSIER = trouves[0].parent
    SORTIES = Path("/kaggle/working/sorties")          # sur Kaggle, on écrit dans /kaggle/working
else:
    DOSSIER = Path(".")
    SORTIES = Path("sorties")                          # en local : sous-dossier sorties/
SORTIES.mkdir(parents=True, exist_ok=True)

CHEMIN_TRAIN = DOSSIER / "train.csv"
CHEMIN_A_ESTIMER = DOSSIER / "a_estimer.csv"

COLS_CAT = [f"cat{i}" for i in range(1, 117)]    # cat1 ... cat116
COLS_CONT = [f"cont{i}" for i in range(1, 15)]   # cont1 ... cont14

GRAINE = 42             # graine du découpage de référence (étape 1.10) et du modèle
NB_REPETITIONS = 20     # nombre de découpages pour la barre de bruit (étape 2.2)
FACTEUR = 2             # un gain doit dépasser FACTEUR x la barre de bruit
BUDGET_SECONDES = 600   # contrainte du sujet : pipeline en moins de 10 minutes sur CPU

print("Dossier des données :", DOSSIER)
print("train.csv existe    :", CHEMIN_TRAIN.exists())
print("a_estimer.csv existe:", CHEMIN_A_ESTIMER.exists())
print("Dossier de sortie   :", SORTIES)

# ── Résultat attendu ──
# Dossier des données : /kaggle/input/datasets/prcieuxfanou/esgc-vak-sujet-04-fanou   (en local : .)
# train.csv existe    : True
# a_estimer.csv existe: True
# Dossier de sortie   : sorties   (ou /kaggle/working/sorties)

Dossier des données : .
train.csv existe    : True
a_estimer.csv existe: True
Dossier de sortie   : sorties


## Bloc 2 : lire et typer les deux fichiers

Mêmes règles qu'à l'étape 1.3 : catégorielles lues en **texte**, le texte `NA` conservé comme modalité.

In [2]:
# ── Ce que fait ce bloc : lire train et a_estimer avec les types de l'étape 1.3 ──
types_lecture = {c: str for c in COLS_CAT}      # les 116 cat lues comme du texte

train = pd.read_csv(CHEMIN_TRAIN, dtype=types_lecture, keep_default_na=False, na_values=[""])
a_estimer = pd.read_csv(CHEMIN_A_ESTIMER, dtype=types_lecture, keep_default_na=False, na_values=[""])

for df in (train, a_estimer):
    df["id"] = df["id"].astype("int64")
    for c in COLS_CONT:
        df[c] = df[c].astype("float64")
train["loss"] = train["loss"].astype("float64")

loss = train["loss"]                              # la cible

print("Forme de train     :", train.shape)
print("Forme de a_estimer :", a_estimer.shape)

# ── Résultat attendu ──
# Forme de train     : (150654, 132)
# Forme de a_estimer : (37664, 131)

Forme de train     : (150654, 132)
Forme de a_estimer : (37664, 131)


## Bloc 3 : encoder les 116 catégorielles en codes ordinaux

Pour chaque colonne, on fabrique **une seule liste de modalités** à partir des deux fichiers (comme à l'étape 1.3), puis on remplace chaque lettre par son **code entier**. Ainsi une lettre a le même code dans `train` et dans `a_estimer`, et une colonne à 326 modalités ne pose aucun problème.

In [3]:
# ── Ce que fait ce bloc : fabriquer les tableaux de variables X_train et X_a_estimer ──
colonnes_codes_train = {}
colonnes_codes_a_estimer = {}

for c in COLS_CAT:
    # union des modalités vues dans les deux fichiers, triée : liste commune
    modalites = sorted(set(train[c].unique()) | set(a_estimer[c].unique()))
    type_commun = pd.CategoricalDtype(categories=modalites, ordered=False)
    # .cat.codes : le code entier de chaque modalité (0, 1, 2, ...)
    colonnes_codes_train[c] = train[c].astype(type_commun).cat.codes
    colonnes_codes_a_estimer[c] = a_estimer[c].astype(type_commun).cat.codes

# Variables explicatives = 116 codes + 14 continues. Ni id, ni loss.
X_train = pd.concat([pd.DataFrame(colonnes_codes_train), train[COLS_CONT]], axis=1)
X_a_estimer = pd.concat([pd.DataFrame(colonnes_codes_a_estimer), a_estimer[COLS_CONT]], axis=1)

# Un code -1 voudrait dire "valeur hors de la liste" : il ne doit pas y en avoir
codes_negatifs = int((X_train[COLS_CAT] < 0).sum().sum() + (X_a_estimer[COLS_CAT] < 0).sum().sum())

print("X_train     :", X_train.shape)
print("X_a_estimer :", X_a_estimer.shape)
print("Codes négatifs (valeurs hors liste) :", codes_negatifs)
print("id ou loss dans les variables :", [c for c in ("id", "loss") if c in X_train.columns])

# ── Résultat attendu ──
# X_train : (150654, 130) ; X_a_estimer : (37664, 130)
# Codes négatifs : 0
# id ou loss dans les variables : []

X_train     : (150654, 130)
X_a_estimer : (37664, 130)
Codes négatifs (valeurs hors liste) : 0
id ou loss dans les variables : []


## Bloc 4 : le plancher et la barre de bruit (rappel des étapes 1.10 et 2.2)

On recalcule ici le plancher (graine 42) et la barre de bruit (20 découpages), pour que ce notebook se suffise à lui-même. Seule la colonne `loss` est utilisée.

In [4]:
# ── Ce que fait ce bloc : recalculer le plancher et la barre de bruit ──
def mae(vraies, prediction):
    """Erreur absolue moyenne : de combien d'euros se trompe-t-on en moyenne ?"""
    return float(np.mean(np.abs(vraies - prediction)))

# 10 tranches de montant : pour stratifier le découpage sur loss
tranches = pd.qcut(loss, q=10, labels=False, duplicates="drop")

def plancher_mediane(graine):
    """Découpage 80/20 stratifié ; MAE de la médiane (apprise sur 80 %) mesurée sur les 20 %."""
    i_app, i_val = train_test_split(train.index, test_size=0.20, random_state=graine, stratify=tranches)
    return mae(loss.loc[i_val], float(loss.loc[i_app].median()))

mae_20 = pd.Series([plancher_mediane(g) for g in range(NB_REPETITIONS)])
barre_bruit = float(mae_20.std(ddof=1))        # écart-type des 20 MAE : la barre de bruit
seuil_gain = FACTEUR * barre_bruit             # gain minimal pour être pris au sérieux

# Le découpage de référence : celui de l'étape 1.10, utilisé pour juger le modèle
indices_app, indices_val = train_test_split(train.index, test_size=0.20, random_state=GRAINE, stratify=tranches)
mediane_app = float(loss.loc[indices_app].median())
plancher_val = mae(loss.loc[indices_val], mediane_app)

print(f"Plancher (médiane, graine {GRAINE}, validation) : {plancher_val:.2f}")
print(f"Barre de bruit (écart-type sur {NB_REPETITIONS} découpages) : {barre_bruit:.2f}")
print(f"Seuil de gain ({FACTEUR} x barre)                 : {seuil_gain:.2f}")
print("Lignes d'apprentissage / validation :", len(indices_app), "/", len(indices_val))

# ── Résultat attendu ──
# Plancher : 1810.40 sur vos données (valeur mesurée à l'étape 1.10).
# Barre de bruit : la valeur mesurée à l'étape 2.2 (à lire, on ne la suppose pas).
# Lignes d'apprentissage / validation : 120523 / 30131

Plancher (médiane, graine 42, validation) : 1810.40
Barre de bruit (écart-type sur 20 découpages) : 6.40
Seuil de gain (2 x barre)                 : 12.81
Lignes d'apprentissage / validation : 120523 / 30131


## Bloc 5 : entraîner le modèle sur 80 %, mesurer sur les 20 % cachés

`HistGradientBoostingRegressor` avec `loss="absolute_error"`. L'arrêt anticipé (`early_stopping`) utilise une petite partie **de l'apprentissage** (10 %), jamais la partie cachée de validation.

In [5]:
# ── Ce que fait ce bloc : entraîner le premier modèle et mesurer sa MAE sur la validation ──
def nouveau_modele():
    """Le modèle, avec ses réglages, défini une seule fois (pour pouvoir le refaire à l'identique)."""
    return HistGradientBoostingRegressor(
        loss="absolute_error",      # on apprend à minimiser l'erreur absolue, comme la métrique
        learning_rate=0.1,
        max_iter=300,               # nombre maximal d'arbres
        early_stopping=True,        # on s'arrête quand l'erreur interne ne baisse plus
        validation_fraction=0.1,    # 10 % de l'APPRENTISSAGE, pas de la validation cachée
        n_iter_no_change=20,
        random_state=GRAINE,
    )

X_app, y_app = X_train.loc[indices_app], loss.loc[indices_app]
X_val, y_val = X_train.loc[indices_val], loss.loc[indices_val]

modele = nouveau_modele()
debut = time.perf_counter()
modele.fit(X_app, y_app)                          # entraînement sur les 80 %
temps_app = time.perf_counter() - debut

pred_val = pd.Series(modele.predict(X_val), index=X_val.index)   # prédictions sur les 20 % cachés
mae_modele = mae(y_val, pred_val)

print(f"Temps d'entraînement : {temps_app:.1f} secondes")
print(f"Nombre d'arbres retenus : {modele.n_iter_}")
print(f"MAE du modèle sur la validation : {mae_modele:.2f}")
print(f"Plancher (médiane)              : {plancher_val:.2f}")

# ── Résultat attendu ──
# Un temps de quelques dizaines de secondes à quelques minutes (à lire, il doit rester sous 10 minutes).
# La MAE du modèle est À MESURER : on s'attend à ce qu'elle batte le plancher, mais on ne le suppose pas.

Temps d'entraînement : 6.2 secondes
Nombre d'arbres retenus : 263
MAE du modèle sur la validation : 1160.21
Plancher (médiane)              : 1810.40


## Bloc 6 : le gain dépasse-t-il le bruit ?

In [6]:
# ── Ce que fait ce bloc : comparer le gain du modèle à la barre de bruit ──
gain = plancher_val - mae_modele                 # euros gagnés par rapport au plancher
gain_pct = gain / plancher_val * 100
gain_en_barres = gain / barre_bruit              # le gain exprimé en nombre de barres de bruit
gain_retenu = bool(gain > seuil_gain)

decision = pd.DataFrame({
    "Quantité": ["Plancher (validation)", "MAE du modèle (validation)", "Gain en euros", "Gain en %",
                 "Gain en barres de bruit", f"Seuil de gain ({FACTEUR} barres)", "Gain démontré ?"],
    "Valeur": [round(plancher_val, 2), round(mae_modele, 2), round(gain, 2), round(gain_pct, 1),
               round(gain_en_barres, 1), round(seuil_gain, 2), "oui" if gain_retenu else "non"],
})
display(decision)

if gain_retenu:
    print("Le premier modèle bat le plancher de plus que le bruit : le gain est démontré.")
else:
    print("Le gain n'est pas démontré : il est dans le bruit, ou le modèle ne bat pas le plancher.")

# ── Résultat attendu ──
# Un tableau de 7 lignes. Le verdict dépend de vos données : lisez-le, ne le supposez pas.

,Quantité,Valeur
0,Plancher (validation),1810.4
1,MAE du modèle (validation),1160.21
2,Gain en euros,650.19
3,Gain en %,35.9
4,Gain en barres de bruit,101.5
5,Seuil de gain (2 barres),12.81
6,Gain démontré ?,oui


Le premier modèle bat le plancher de plus que le bruit : le gain est démontré.


## Bloc 7 : analyse d'erreurs

Où le modèle gagne-t-il, et où se trompe-t-il ? On regarde l'erreur par tranche de montant réel, les dix sinistres les plus lourds, et le partage entre sous-estimation et surestimation.

In [7]:
# ── Ce que fait ce bloc : analyser les erreurs du modèle sur la validation ──
erreur_modele = (y_val - pred_val).abs()
erreur_plancher = (y_val - mediane_app).abs()

# 1) Par tranche de montant RÉEL (déciles de la validation)
tranche_val = pd.qcut(y_val, q=10, labels=False, duplicates="drop")
par_tranche = pd.DataFrame({
    "tranche": tranche_val,
    "montant_moyen": y_val,
    "MAE_modele": erreur_modele,
    "MAE_plancher": erreur_plancher,
}).groupby("tranche").mean().round(2)
par_tranche["gain"] = (par_tranche["MAE_plancher"] - par_tranche["MAE_modele"]).round(2)
print("Erreur par tranche de montant réel (0 = petits sinistres, 9 = gros sinistres) :")
display(par_tranche)

# 2) Les 10 sinistres les plus lourds de la validation
lourds = y_val.nlargest(10).to_frame("loss_reel")
lourds["prediction_modele"] = pred_val.loc[lourds.index].round(2)
lourds["erreur"] = (lourds["loss_reel"] - lourds["prediction_modele"]).round(2)
print("Les 10 sinistres les plus lourds de la validation :")
display(lourds)

# 3) Sous-estimation ou surestimation ?
ecart = y_val - pred_val                        # positif : le vrai montant dépasse la prédiction
sous = ecart[ecart > 0]
sur = -ecart[ecart < 0]
print(f"Sinistres sous-estimés : {len(sous) / len(ecart) * 100:.1f} % ; euros de sous-estimation : {sous.sum():,.0f}")
print(f"Sinistres surestimés   : {len(sur) / len(ecart) * 100:.1f} % ; euros de surestimation   : {sur.sum():,.0f}")

# ── Résultat attendu ──
# Des tableaux à LIRE : quelles tranches gagnent, lesquelles perdent par rapport au plancher,
# et de combien le modèle se trompe sur les plus gros sinistres.

Erreur par tranche de montant réel (0 = petits sinistres, 9 = gros sinistres) :


,montant_moyen,MAE_modele,MAE_plancher,gain
tranche,,,,
0,593.67,696.70,1521.91,825.21
1,914.54,612.97,1201.04,588.07
2,1206.58,615.66,909.00,293.34
3,1526.61,586.30,588.97,2.67
4,1903.41,638.84,212.17,-426.67
5,2373.45,773.88,257.87,-516.01
6,2987.62,950.35,872.04,-78.31
7,3878.62,1296.39,1763.04,466.65
8,5308.31,1738.56,3192.73,1454.17


Les 10 sinistres les plus lourds de la validation :


,loss_reel,prediction_modele,erreur
60955,37313.28,9585.46,27727.82
135211,33071.97,15505.06,17566.91
125349,32447.93,11733.60,20714.33
42080,32099.24,16551.62,15547.62
106511,31710.15,8688.15,23022.00
106488,31657.86,14964.91,16692.95
55252,31302.18,17407.25,13894.93
132252,30532.31,3781.45,26750.86
103529,30377.23,14528.91,15848.32
30110,29769.75,14488.78,15280.97


Sinistres sous-estimés : 49.7 % ; euros de sous-estimation : 22,227,937
Sinistres surestimés   : 50.3 % ; euros de surestimation   : 12,730,319


## Bloc 8 : le journal des essais

Le sujet demande de **consigner tous les essais, y compris ceux qui échouent**, avec leur score. On écrit une ligne pour cet essai ; les notebooks suivants en ajouteront.

In [8]:
# ── Ce que fait ce bloc : noter l'essai 1 dans un fichier journal_essais.csv ──
chemin_journal = SORTIES / "journal_essais.csv"

ligne = pd.DataFrame([{
    "essai": "1_hgb_ordinal_absolute_error",
    "modele": "HistGradientBoostingRegressor",
    "encodage_cat": "codes ordinaux (liste commune train + a_estimer)",
    "perte": "absolute_error",
    "mae_validation": round(mae_modele, 2),
    "plancher_validation": round(plancher_val, 2),
    "gain": round(gain, 2),
    "gain_demontre": "oui" if gain_retenu else "non",
    "temps_secondes": round(temps_app, 1),
}])

if chemin_journal.exists():
    journal = pd.read_csv(chemin_journal)
    journal = journal[journal["essai"] != ligne.loc[0, "essai"]]     # on remplace l'essai s'il existe déjà
    journal = pd.concat([journal, ligne], ignore_index=True)
else:
    journal = ligne

journal.to_csv(chemin_journal, index=False)
print("Journal écrit :", chemin_journal, "|", len(journal), "essai(s)")
display(journal)

# ── Résultat attendu ──
# Un fichier journal_essais.csv avec une ligne pour l'essai 1.

Journal écrit : sorties/journal_essais.csv | 1 essai(s)


,essai,modele,encodage_cat,perte,mae_validation,plancher_validation,gain,gain_demontre,temps_secondes
0,1_hgb_ordinal_absolute_error,HistGradientBoostingRegressor,codes ordinaux (liste commune train + a_estimer),absolute_error,1160.21,1810.4,650.19,oui,6.2


## Bloc 9 : entraîner sur tout `train` et estimer les 37 664 sinistres

Le modèle final voit **tout** `train` (la partie cachée comprise), puis prédit `a_estimer`. On écrit `soumission_modele1.csv` au format `id,loss` et on vérifie ce format.

In [9]:
# ── Ce que fait ce bloc : refaire le modèle sur 100 % de train, prédire a_estimer, écrire le fichier ──
modele_final = nouveau_modele()
debut = time.perf_counter()
modele_final.fit(X_train, loss)                     # tout train, avec les mêmes réglages
temps_final = time.perf_counter() - debut

pred_brute = modele_final.predict(X_a_estimer)
nb_negatives = int((pred_brute < 0).sum())          # un montant négatif est impossible
pred_finale = np.maximum(pred_brute, 0.0)           # on ramène à 0 les éventuelles prédictions négatives

soumission = pd.DataFrame({"id": a_estimer["id"].to_numpy(), "loss": pred_finale})
chemin_soumission = SORTIES / "soumission_modele1.csv"
soumission.to_csv(chemin_soumission, index=False)

print(f"Temps d'entraînement sur tout train : {temps_final:.1f} secondes")
print("Prédictions négatives ramenées à 0 :", nb_negatives)
print("Fichier écrit :", chemin_soumission, "|", len(soumission), "lignes | colonnes", list(soumission.columns))
display(soumission.head())

# ── Résultat attendu ──
# 37664 lignes, colonnes ['id', 'loss'], aucune valeur manquante, aucun montant négatif.

Temps d'entraînement sur tout train : 8.2 secondes
Prédictions négatives ramenées à 0 : 1
Fichier écrit : sorties/soumission_modele1.csv | 37664 lignes | colonnes ['id', 'loss']


,id,loss
0,144215,2034.785196
1,348120,2524.878261
2,108743,1361.138798
3,98241,1522.343482
4,87110,2028.068620


## Bloc 10 : tableau récapitulatif de l'étape 2.5

In [10]:
# ── Ce que fait ce bloc : contrôles internes, puis résultat à retenir ──
def joli(v):
    """Affiche 150654 plutôt que 150654.00."""
    return str(int(v)) if float(v).is_integer() else str(round(float(v), 2))

temps_total = temps_app + temps_final     # les deux entraînements (la lecture des fichiers n'est pas comptée)

controles = [
    ("Lignes de train",                                      150654, len(train)),
    ("Variables explicatives",                               130,    X_train.shape[1]),
    ("id ou loss dans les variables (nombre)",               0,      sum(c in X_train.columns for c in ("id", "loss"))),
    ("Codes négatifs (valeurs hors liste)",                  0,      codes_negatifs),
    ("id communs apprentissage / validation",                0,      len(set(indices_app) & set(indices_val))),
    ("Lignes apprentissage + validation",                    150654, len(indices_app) + len(indices_val)),
    ("Entraînements sous 10 minutes (1 = oui)",              1,      int(temps_total < BUDGET_SECONDES)),
    ("Lignes de la soumission",                              37664,  len(soumission)),
    ("id en double dans la soumission",                      0,      int(soumission["id"].duplicated().sum())),
    ("Mêmes id que a_estimer (1 = oui)",                     1,      int(set(soumission["id"]) == set(a_estimer["id"]))),
    ("Valeurs manquantes dans la soumission",                0,      int(soumission["loss"].isna().sum())),
    ("Montants négatifs dans la soumission",                 0,      int((soumission["loss"] < 0).sum())),
]

recap = pd.DataFrame(controles, columns=["Contrôle", "Attendu", "Obtenu"])
recap["Verdict"] = recap.apply(
    lambda ligne_: "OK" if ligne_["Obtenu"] == ligne_["Attendu"] else "ATTENTION", axis=1
)
recap["Attendu"] = recap["Attendu"].map(joli)
recap["Obtenu"] = recap["Obtenu"].map(joli)
display(recap)

# Le résultat à retenir
resume = pd.DataFrame({
    "Quantité": ["Plancher (médiane, validation)", "MAE du premier modèle (validation)", "Gain",
                 "Barre de bruit", f"Seuil de gain ({FACTEUR} barres)", "Temps total d'entraînement (s)"],
    "Valeur": [round(plancher_val, 2), round(mae_modele, 2), round(gain, 2),
               round(barre_bruit, 2), round(seuil_gain, 2), round(temps_total, 1)],
})
display(resume)

print("Gain démontré (au-dessus du bruit) :", "oui" if gain_retenu else "non")

if (recap["Verdict"] == "OK").all():
    print("Étape 2.5 validée : le premier modèle est entraîné, mesuré et la soumission est au bon format.")
else:
    print("ATTENTION : au moins un contrôle ne passe pas. Regarder la colonne Verdict.")

# ── Résultat attendu ──
# Un tableau de 12 contrôles, tous en OK. Le résumé donne la MAE du modèle contre le plancher :
# ces valeurs sont à lire sur vos données, pas à supposer.

,Contrôle,Attendu,Obtenu,Verdict
0,Lignes de train,150654,150654,OK
1,Variables explicatives,130,130,OK
2,id ou loss dans les variables (nombre),0,0,OK
3,Codes négatifs (valeurs hors liste),0,0,OK
4,id communs apprentissage / validation,0,0,OK
5,Lignes apprentissage + validation,150654,150654,OK
6,Entraînements sous 10 minutes (1 = oui),1,1,OK
7,Lignes de la soumission,37664,37664,OK
8,id en double dans la soumission,0,0,OK
9,Mêmes id que a_estimer (1 = oui),1,1,OK


,Quantité,Valeur
0,"Plancher (médiane, validation)",1810.40
1,MAE du premier modèle (validation),1160.21
2,Gain,650.19
3,Barre de bruit,6.40
4,Seuil de gain (2 barres),12.81
5,Temps total d'entraînement (s),14.40


Gain démontré (au-dessus du bruit) : oui
Étape 2.5 validée : le premier modèle est entraîné, mesuré et la soumission est au bon format.
